In [2]:
pip install torch

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 16.0 MB/s  0:00:05m0:00:0100:01
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl (12 kB)
  Attempting uninstall: setuptools━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [sympy]
    Found existing installation: setuptools 82.0.1━━━━━━━━━━━━ 1/7 [sympy]
    Uninstall

In [1]:
# developing some examples of GPT2 

from transformers import GPT2Tokenizer, set_seed

text = (
    "Previously, the customer has bought: Better Man. Gold. Gold. Invitation Only. "
    "I Want You Remastered. Greatest Love Songs. Chicago '85 The Movie. "
    "In the future, the customer wants to buy Perfect Moment"
)

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

enc = tokenizer(text, padding=True, truncation=True, max_length=50, return_tensors="pt")
ids = enc["input_ids"][0].tolist()

print("input_ids:", ids)
print("tokens:", tokenizer.convert_ids_to_tokens(ids))
print("decoded:", tokenizer.decode(ids))

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


input_ids: [36837, 11, 262, 6491, 468, 5839, 25, 11625, 1869, 13, 3561, 13, 3561, 13, 10001, 3780, 5514, 13, 314, 16168, 921, 22773, 13, 33575, 5896, 31772, 13, 4842, 705, 5332, 383, 15875, 13, 554, 262, 2003, 11, 262, 6491, 3382, 284, 2822, 16374, 29278]
tokens: ['Previously', ',', 'Ġthe', 'Ġcustomer', 'Ġhas', 'Ġbought', ':', 'ĠBetter', 'ĠMan', '.', 'ĠGold', '.', 'ĠGold', '.', 'ĠInv', 'itation', 'ĠOnly', '.', 'ĠI', 'ĠWant', 'ĠYou', 'ĠRemastered', '.', 'ĠGreatest', 'ĠLove', 'ĠSongs', '.', 'ĠChicago', "Ġ'", '85', 'ĠThe', 'ĠMovie', '.', 'ĠIn', 'Ġthe', 'Ġfuture', ',', 'Ġthe', 'Ġcustomer', 'Ġwants', 'Ġto', 'Ġbuy', 'ĠPerfect', 'ĠMoment']
decoded: Previously, the customer has bought: Better Man. Gold. Gold. Invitation Only. I Want You Remastered. Greatest Love Songs. Chicago '85 The Movie. In the future, the customer wants to buy Perfect Moment


In [22]:
# grab example of how the generated queries would look for example user

from transformers import GPT2LMHeadModel, GPT2Tokenizer
from config import GlobalConfig
from gpt4rec.model import GPT4RecCandidateRanker, GPT4RecGenerationModel
import torch

config = GlobalConfig()
args = config.model_namespace("gpt4rec")
lm = GPT2LMHeadModel.from_pretrained("distilgpt2")

class TmpCfg:
    pass
tmp = TmpCfg()
tmp.num_users = 999 # place holder
tmp.num_items = 999 # place holder
tmp.vocab_size = tokenizer.vocab_size
tmp.n_embd = lm.config.n_embd # in order to deal with shape mismatch between lm and gpt4rec model
tmp.initializer_range = args.initializer_range

model = GPT4RecGenerationModel(tmp, lm)
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
device = torch.device("cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu")

val_prompt = ["Previously, the customer has bought: Better Man. Gold. Gold. Invitation Only. I Want You Remastered. Greatest Love Songs. Chicago ’85 The Movie. Perfect Moment. In the future, the customer wants to buy"]

# tokenize
enc = tokenizer(
            val_prompt,
            padding=True,
            truncation=True,
            max_length=int(getattr(args, "maxlen", 128)),
            return_tensors="pt",
        )
       
input_ids = enc["input_ids"]
attention_mask = enc["attention_mask"]
print("Attention mask:", attention_mask)
prompt_len = int(input_ids.shape[1])

gen = model.generate_queries(
    input_ids,
    attention_mask=attention_mask,
    num_beams=args.num_beams,
    num_return_sequences=args.num_queries_per_user,
    max_new_tokens=args.max_query_tokens,
)

print(gen) # generated queries

for row in gen.tolist():
    print(tokenizer.decode(row[prompt_len:], skip_special_tokens=True))

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 19323.90it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[36837,    11,   262,  6491,   468,  5839,    25, 11625,  1869,    13,
          3561,    13,  3561,    13, 10001,  3780,  5514,    13,   314, 16168,
           921, 22773,    13, 33575,  5896, 31772,    13,  4842,   564,   247,
          5332,   383, 15875,    13, 16374, 29278,    13,   554,   262,  2003,
            11,   262,  6491,  3382,   284,  2822,   257,   649,  5062,    13,
           198,   198,   198,   198,   198,   198,   198,   198,   198,   198,
           198,   198],
        [36837,    11,   262,  6491,   468,  5839,    25, 11625,  1869,    13,
          3561,    13,  3561,    13, 10001,  3780,  5514,    13,   314, 16168,
           921, 22773,    13, 33575,  5896, 31772,    13,  4842,   564,   247,
          5332,   383, 15875,    13, 16374, 29278,    13,   554,   262,  2003,
            

In [ ]:
from gpt4rec.build_raptor import load_raptor_tree, SBertEmbeddingModel
import sys
from pathlib import Path

import pickle

import re

ITEM_RE = re.compile(r"\[ITEM\s+(\d+)\]")

def build_parent_map(tree):
    """child_index -> parent_index"""
    parent_of = {}
    for node in tree.all_nodes.values():
        for child_idx in node.children:
            parent_of[child_idx] = node.index
    return parent_of

def layer_of(tree, node_index):
    for layer, nodes in tree.layer_to_nodes.items():
        if any(n.index == node_index for n in nodes):
            return layer
    return None

def climb_from_leaf(tree, leaf_index=None, preview_chars=500):

    parent_of = build_parent_map(tree)

    # pick one leaf (first leaf if you don't pass an index)
    if leaf_index is None:
        leaf = next(iter(tree.leaf_nodes.values()))
        leaf_index = leaf.index
    else:
        leaf = tree.all_nodes[leaf_index]

    path = [leaf]
    cur = leaf_index
    while cur in parent_of:
        cur = parent_of[cur]
        path.append(tree.all_nodes[cur])

    print(f"Path length: {len(path)} nodes (leaf -> root)")

    for node in path:
        layer = layer_of(tree, node.index)
        m = ITEM_RE.search(node.text)
        item_tag = f" [item_id={m.group(1)}]" if m else ""
        print(f"\n--- Layer {layer} (node {node.index}){item_tag} ---")
        print(node.text[:preview_chars])
        if len(node.text) > preview_chars:
            print("...")

    return path

path = climb_from_leaf(tree) # loaded from saved RAPTOR tree



Path length: 6 nodes (leaf -> root)

--- Layer 0 (node 3) [item_id=26] ---
B015WC2KEY [ITEM 26] B00X8UKN42 [ITEM 27] B01IS2LXBG [ITEM 28] B01D3N6TKA [ITEM 29] B01LWY8995 [ITEM 30] B00I3MQNWG [ITEM 31] Rockin' In Rhythm: A Duke Ellington Tribute [ITEM 32] Guitar Prayer [ITEM 33]

--- Layer 1 (node 40842) ---
B015WC2KEY [ITEM 26: B00X8UKN42: B01IS2LXBG: Guitar Prayer . B01D3N6TKA: A Duke Ellington Tribute . B07787GZGP: 50th anniversary of the Fender Stratocaster . B004436E5Y: The Comedy Collection .

--- Layer 2 (node 43002) [item_id=175509] ---
The Cranberries: No Need To Argue (180g) Vinyl LP [ITEM 175509] Jug Band Music [itEM 175510] Relax Your Mind [ITME 175511] The Folk Blues of Dave Van Ronk [ITIM 175512] B01EGH2CXU: Shaft Deluxe (ITEM 109954) Be For Real: The P I R  Recordings 1972-1975 . B0050YLT2

--- Layer 3 (node 43180) [item_id=175509] ---
KISS – The Solo Albums 40th Anniversary Collection . KISS . The Beatle White Album is The Beatles White Album . Rush - Replay: Boxed Set i